In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Navigate to your project folder
# CHECK: Make sure 'ai-error-tutor' matches exactly what you named the folder in Drive
project_path = '/content/drive/MyDrive/ai-error-tutor'

try:
    os.chdir(project_path)
    print(f"✅ Moved to: {os.getcwd()}")
    
    # 3. Move into 'notebooks' so your relative paths (../data) work correctly
    os.chdir('notebooks')
    print(f"✅ Final working directory: {os.getcwd()}")
except FileNotFoundError:
    print("❌ Error: Could not find the folder. Check your Google Drive structure!")

In [ ]:
# Install dependencies
!pip install transformers datasets torch pandas scikit-learn rouge-score

import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Check if GPU is available (Recommended for faster training)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# OPTION 1: Create sample data (Use this to test immediately)
# data = {
#     'error_type': ['NameError', 'TypeError', 'SyntaxError', 'IndexError', 'KeyError'] * 20,
#     'raw_error': [
#         "NameError: name 'prnt' is not defined",
#         "TypeError: unsupported operand type(s) for +: 'int' and 'str'",
#         "SyntaxError: invalid syntax",
#         "IndexError: list index out of range",
#         "KeyError: 'missing_key'"
#     ] * 20,
#     'code_context': [
#         "prnt('Hello')",
#         "x = 5 + 'hello'",
#         "def foo()\n    pass",
#         "lst = [1,2]; lst[10]",
#         "d = {}; d['missing_key']"
#     ] * 20,
#     'friendly_explanation': [
#         "You have a typo - 'prnt' should be 'print'.",
#         "You can't add a number and text directly.",
#         "Missing colon after function definition.",
#         "You're accessing an index that doesn't exist.",
#         "The key doesn't exist in the dictionary."
#     ] * 20,
#     'suggested_fix': [
#         "print('Hello')",
#         "x = str(5) + 'hello'",
#         "def foo():",
#         "lst = [1,2] # Check index first",
#         "d.get('missing_key')"
#     ] * 20
# }

# df = pd.DataFrame(data)

# OPTION 2: Load from CSV (If you upload your file to Colab files tab)
df = pd.read_csv('../data/error_dataset.csv')

# Split data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

In [ ]:
class ErrorDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input=256, max_target=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Construct the input prompt
        input_text = f"explain error: {row['error_type']} | message: {row['raw_error']} | code: {row['code_context']}"
        target_text = f"{row['friendly_explanation']} FIX: {row['suggested_fix']}"

        # Tokenize Input
        inputs = self.tokenizer(
            input_text, 
            max_length=self.max_input,
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )

        # Tokenize Target (Label)
        targets = self.tokenizer(
            target_text, 
            max_length=self.max_target,
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': targets['input_ids'].squeeze()
        }

# Initialize Tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Create Datasets (Now properly placed outside the class)
train_dataset = ErrorDataset(train_df, tokenizer)
val_dataset = ErrorDataset(val_df, tokenizer)

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

training_args = TrainingArguments(
    output_dir='../results',
    num_train_epochs=5,           # Reduced to 5 for quicker testing
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available()  # Only use mixed precision if GPU is available
)

trainer = Trainer(
    model=model,
    args=training_args,         
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Start Training
trainer.train()

In [ ]:
save_path = '../models/error_tutor_model'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Optional: Download to your local machine (uncomment if needed)
# from google.colab import files
# !zip -r error_tutor_model.zip ./error_tutor_model
# files.download('error_tutor_model.zip')

In [ ]:
def explain_error(error_type, message, code):
    # Prepare input text matching the training format
    input_text = f"explain error: {error_type} | message: {message} | code: {code}"
    
    # Move inputs to the same device as the model (GPU or CPU)
    inputs = tokenizer(input_text, return_tensors='pt', max_length=256, truncation=True)
    input_ids = inputs['input_ids'].to(model.device)
    
    # Generate prediction
    outputs = model.generate(
        input_ids=input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test Case
result = explain_error(
    "NameError", 
    "name 'prnt' is not defined", 
    "prnt('Hello')"
)

print("--- AI Tutor Response ---")
print(result)